# Phase 2 L-NER with c-1 (decoder-only): full fine-tuning (E1) and LoRA (E2)

**Setup:** Add Input → Your Work → phase-1-c-1 (decoder-only). Accelerator GPU T4 x2, Internet on.

Run order: Cell 0 → A → B → C → D → E0 → E1 → E2. E2 reloads the Phase 1 weights itself, so it can run straight after E1 (or straight after E0 if you skip E1). To run E1 again, re-run Cell A and E0 first.

c-1 is a causal decoder, so windows follow the c-5 rules (left context, only the last STRIDE tokens of a window are labelled, right padding), not the c-3 encoder rules.

All TODOs are filled with your own choices: E1 from your c-3 full fine-tune, E2 from your c-6 LoRA.

## Cell 0: classes from your Phase 1 c-1 notebook (cells 1, 8, 21, in that order)

In [ ]:
# --- Phase 1 c-1 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


In [2]:
# --- Phase 1 c-1 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok


In [3]:
# --- Phase 1 c-1 notebook, cell 21 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


In [4]:
# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell A: find and load the c-1 checkpoint and tokenizer

In [5]:
!find /kaggle/input -path "*phase-1-c-1*" \( -name "*.pt" -o -name "*.pkl" \) -exec ls -la {} \;


-rw-r--r-- 1 nobody nogroup 429235 Oct 11 15:10 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/bpe_tokenizer.pkl
-rw-r--r-- 1 nobody nogroup 429235 Oct 11 15:10 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl
-rw-r--r-- 1 nobody nogroup 1331870311 Oct 11 15:11 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/ckpt.pt
-rw-r--r-- 1 nobody nogroup 443954142 Oct 11 15:10 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/model_final.pt


In [6]:
# Expect <All keys matched successfully>, 110.98M params, vocab 10000, d_model 1024, 8 layers, max_seq_len 2048
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only"   # fix to what find showed
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl", "rb"))
ck = torch.load(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt", map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| layers", cfg.n_layers, "| max_seq_len", cfg.max_seq_len,
      "| norm_first", cfg.norm_first, "| dropout", cfg.dropout)


<All keys matched successfully>
params 110.979072 M | vocab 10000 | d_model 1024 | layers 8 | max_seq_len 2048 | norm_first False | dropout 0.1


## Cell B: per-token features from the decoder (sanity check should print max diff 0.0)

In [7]:
def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024), LM head skipped

# Sanity check: the LM head on these features must give the full model's logits
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f = features(idx); diff = (lm.head(f) - lm(idx)).abs().max().item()
print("max diff", diff, tuple(f.shape))   # expect 0.0, (1, 64, 1024)


max diff 0.0 (1, 64, 1024)


## Cell C: Phase 2 data

In [8]:
from huggingface_hub import hf_hub_download
def get(name):
    p = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_2_LNER/{name}", repo_type="dataset")
    return json.load(open(p)) if p.endswith(".json") else open(p).read().split()
f1_text, f1_gold = get("Train/fold1-text.json"), get("Train/fold1-gold.json")
f2_text, f2_gold = get("Train/fold2-text.json"), get("Train/fold2-gold.json")
test_text = get("Test/fold3-text.json")
LABELS = get("Train/labels.txt")
print(len(f1_text), len(f2_text), len(test_text), LABELS)
k = next(iter(f1_gold)); print(k, f1_gold[k][:3])     # check the annotation format before going on


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

35 35 35 ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
115651329 [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana']]


## Cell D: character offsets + round-trip check

In [9]:
SPLIT = re.compile(r'\s*\S+|\s+')

def encode_with_offsets(text):
    ids, offs = [], []
    for m in SPLIT.finditer(text):
        chunk, cs = m.group(), m.start()
        b2c = []
        for ci, ch in enumerate(chunk):
            b2c += [ci] * len(ch.encode("utf-8"))
        b = 0
        for t in tok.encode(chunk):
            n = len(tok.id_to_bytes[t])
            s, e = cs + b2c[b], cs + b2c[b + n - 1] + 1
            ids.append(t); offs.append((s, e)); b += n
    return ids, offs

def content_start(text, s, e):            # skip the leading space a token carries
    while s < e and text[s].isspace(): s += 1
    return s

def spans(gold_doc):                      # -> list of (start, end, label)
    return [(int(g[0]), int(g[1]), g[2]) for g in gold_doc]

# Check 1: token ids match the Phase 1 tokenizer exactly
t = f1_text[k]; ids, offs = encode_with_offsets(t)
assert ids == tok.encode(t), "chunked encoding differs from tok.encode"

# Check 2: gold -> token BIO -> back to spans
def to_bio(text, offs, ents):
    tags = ["O"] * len(offs)
    for s, e, lab in ents:
        first = True
        for i, (ts, te) in enumerate(offs):
            cs_ = content_start(text, ts, te)
            inside_ws = cs_ == te and s < ts and te < e   # whitespace-only token strictly inside the entity
            if (cs_ < e and te > s and cs_ < te) or inside_ws:   # token overlaps the entity
                tags[i] = ("B-" if first else "I-") + lab; first = False
    return tags

def from_bio(text, offs, tags):
    out, cur = [], None
    for (ts, te), tg in zip(offs, tags):
        if tg.startswith("B-") or (tg.startswith("I-") and (cur is None or cur[2] != tg[2:])):
            if cur: out.append(tuple(cur))
            cur = [content_start(text, ts, te), te, tg[2:]]
        elif tg.startswith("I-"):
            cur[1] = te
        else:
            if cur: out.append(tuple(cur)); cur = None
    if cur: out.append(tuple(cur))
    return out

hit = tot = 0; bad = []
for texts, golds in [(f1_text, f1_gold), (f2_text, f2_gold)]:
    for d in golds:
        text = texts[d]; _, offs = encode_with_offsets(text)
        g = set(spans(golds[d])); r = set(from_bio(text, offs, to_bio(text, offs, g)))
        hit += len(g & r); tot += len(g)
        bad += [(text[s:e], lab, d) for s, e, lab in g - r][:3]
print(f"round-trip exact: {hit}/{tot} = {hit/tot:.2%}"); print(bad[:15])


round-trip exact: 6186/8393 = 73.70%
[('IPC', 'STAT', '115651329'), ('Ramesh Kumar', 'WIT', '115651329'), ('Vibha Datta Makhija', 'A.COUNSEL', '115651329'), ('C.A.No.4085 of 2010', 'CASENO', '37849282'), ('Societies Registration Act, 1860', 'STAT', '37849282'), ('14.05.2010', 'DATE', '37849282'), ('TP Act', 'STAT', '975074'), ('Civil Court', 'COURT', '975074'), ('T.P. Act', 'STAT', '975074'), ('Ramveer', 'APP', '189525449'), ('IPC', 'STAT', '189525449'), ('9.8.1993', 'DATE', '189525449'), ('Union of India', 'RESP', '736324'), ('SLP No. 2377/2001', 'CASENO', '736324'), ('O.A. No.93 of 1999', 'CASENO', '736324')]


## Cell E0: shared plumbing (c-1 decoder path, causal windows, training loop, scorer, submission)

In [10]:
# =====================================================================================
# CELL E0 — shared plumbing for the c-1 Phase 2 experiments
# Run once after Cells 0, A, B, C, D. Then run the technique cell (E1 full fine-tuning).
#
# c-1 is a CAUSAL decoder (each token sees only tokens to its left), so the c-5 rules
# apply, not the c-3 encoder rules:
#   - windows of <= WINDOW tokens every STRIDE tokens; every window after the first
#     carries WINDOW-STRIDE tokens of left context, and only its last STRIDE tokens get
#     labels / predictions, so every token is labelled exactly once and always has
#     left context (no multiple-of-4 rule here);
#   - batches are RIGHT-padded with <pad> (safe: a causal token never sees the pads to
#     its right), and padded positions get no loss and no prediction.
# Needs from earlier cells: lm, tok, cfg, device, features, LABELS,
#   f1_text, f1_gold, f2_text, f2_gold, test_text, encode_with_offsets, to_bio, from_bio, spans
#
# Nothing in this cell is a model design or a hyperparameter choice. It only does:
# tags, windows, batching, the c-1 decoder forward path, scoring, the training loop,
# prediction and packaging.
# =====================================================================================
import math, time, json, zipfile, random
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm   # every loop below that takes time shows a progress bar

# -------------------------------------------------------------------------------------
# Tags: O + B-/I- for each of the 12 labels -> 25 tags
# -------------------------------------------------------------------------------------
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
N_TAGS = len(TAGS)
IGNORE = -100          # positions with no loss (left context, padding)
FEAT_DIM = cfg.d_model # 1024 for c-1
PAD_ID = 0             # "<pad>" in SPECIAL_TOKENS

# Full fine-tuning of 8 blocks x 2048 tokens is memory-heavy on a T4 (16 GB).
# True = recompute each block's activations in the backward pass (less memory, ~30% slower).
# This does not change the maths, only memory vs speed.
GRAD_CHECKPOINT = False


# -------------------------------------------------------------------------------------
# c-1 decoder path (identical to Cell B features(), but on any backbone object)
# -------------------------------------------------------------------------------------
def decoder_forward(backbone, idx):
    T = idx.size(1)
    assert T <= backbone.cfg.max_seq_len, "window longer than 2048"
    x = backbone.tok_emb(idx)
    if backbone.cfg.pos_emb_type == "sinusoidal":
        x = x + backbone.pos_emb[:T, :]
    for blk in backbone.blocks:
        if GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training:
            x = checkpoint(blk, x, use_reentrant=False)
        else:
            x = blk(x)
    return backbone.ln_f(x)                     # (B, T, 1024); LM head skipped


class NERModel(nn.Module):
    """backbone (c-1 decoder path) -> head -> per-token tag logits."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head
    def forward(self, idx):
        return self.head(decoder_forward(self.backbone, idx))   # (B, T, 25)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Windows (causal rule, same as c-5)
# -------------------------------------------------------------------------------------
def make_windows(n, window, stride):
    """List of (start, end, first): first = index inside the window of the first labelled token."""
    out, s = [], 0
    while True:
        e = min(s + window, n)
        first = 0 if s == 0 else window - stride
        out.append((s, e, first))
        if e >= n:
            return out
        s += stride


def build_examples(texts, golds, window, stride):
    """One example per window. golds: dict, or None for the test set."""
    examples, cache = [], {}
    for doc_id, text in tqdm(texts.items(), desc="tokenising + windowing", leave=False):
        ids, offs = encode_with_offsets(text)
        cache[doc_id] = (text, offs)
        if not ids:                     # empty document: no windows, nothing to tag
            continue
        tag_ids = None
        if golds is not None:
            tag_ids = [TAG2ID[t] for t in to_bio(text, offs, spans(golds.get(doc_id, [])))]
        for s, e, first in make_windows(len(ids), window, stride):
            ex = {"doc": doc_id, "start": s, "first": first, "ids": ids[s:e], "labels": None}
            if tag_ids is not None:
                ex["labels"] = [IGNORE] * first + tag_ids[s + first:e]
            examples.append(ex)
    return examples, cache


def make_batches(examples, batch_size, shuffle):
    """Batches of windows of similar length (less padding). Shuffled order when training."""
    exs = list(examples)
    if shuffle:
        random.shuffle(exs)
    out, pool = [], batch_size * 50
    for i in range(0, len(exs), pool):          # sort by length inside pools of 50 batches
        chunk = sorted(exs[i:i + pool], key=lambda ex: len(ex["ids"]))
        out += [chunk[j:j + batch_size] for j in range(0, len(chunk), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def to_tensors(batch):
    """Right-pad to the longest window in the batch."""
    T = max(len(ex["ids"]) for ex in batch)
    ids = torch.full((len(batch), T), PAD_ID, dtype=torch.long)
    for i, ex in enumerate(batch):
        ids[i, :len(ex["ids"])] = torch.tensor(ex["ids"])
    b = {"ids": ids.to(device)}
    if batch[0]["labels"] is not None:
        lab = torch.full((len(batch), T), IGNORE, dtype=torch.long)
        for i, ex in enumerate(batch):
            lab[i, :len(ex["labels"])] = torch.tensor(ex["labels"])
        b["labels"] = lab.to(device)
    return b


def ce_loss(logits, labels, class_weights=None):
    """Plain token cross-entropy on gold tags (ignores IGNORE positions)."""
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1),
                           weight=class_weights, ignore_index=IGNORE)


# -------------------------------------------------------------------------------------
# Prediction + strict scoring
# -------------------------------------------------------------------------------------
TRAIL_CHARS = ",.;:"   # punctuation that BPE glues onto an entity's last token but gold spans exclude;
                       # set to "" to switch off.

def decode_spans(text, offs, tags):
    """from_bio, then drop TRAIL_CHARS from the end of each span (decoding rule, no model involved)."""
    out = []
    for s, e, l in from_bio(text, offs, tags):
        while TRAIL_CHARS and e > s and text[e - 1] in TRAIL_CHARS:
            e -= 1
        if e > s:                       # a span that was only punctuation vanishes
            out.append((s, e, l))
    return out


@torch.no_grad()
def predict(model, examples, cache, batch_size):
    """{doc_id: [(start_char, end_char, label), ...]}; each token predicted by the one window that labels it."""
    model.eval()
    doc_tags = {d: ["O"] * len(cache[d][1]) for d in cache}
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(to_tensors(batch)["ids"]).argmax(-1).cpu()
        for b, ex in enumerate(batch):
            for j in range(ex["first"], len(ex["ids"])):
                doc_tags[ex["doc"]][ex["start"] + j] = TAGS[pred[b, j]]
    return {d: decode_spans(cache[d][0], cache[d][1], doc_tags[d]) for d in cache}


def strict_macro_f1(pred, gold):
    """Exact (start, end, label) match; per-label P/R/F1, macro over labels seen in gold or
    predictions. Cross-check with the official script before trusting small differences."""
    tp = {l: 0 for l in LABELS}; fp = dict(tp); fn = dict(tp)
    for d in gold:
        g, p = set(spans(gold[d])), set(pred.get(d, []))
        for s, e, l in p & g: tp[l] += 1
        for s, e, l in p - g: fp[l] = fp.get(l, 0) + 1
        for s, e, l in g - p: fn[l] += 1
    used = [l for l in LABELS if tp[l] + fp[l] + fn[l] > 0]
    per = {}
    for l in used:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = (P, R, 2 * P * R / (P + R) if P + R else 0.0)
    return (sum(v[2] for v in per.values()) / len(per) if per else 0.0), per


# -------------------------------------------------------------------------------------
# Training loop. loss_fn(logits, batch_tensors) is supplied by the technique cell.
# lr_lambda: optional step -> multiplier (your schedule), or None for constant LR.
# head_lr: optional separate LR for model.head (None = same LR as the backbone).
# -------------------------------------------------------------------------------------
def train(model, train_ex, val_ex, val_cache, val_gold, loss_fn, *, lr, epochs, batch_size,
          weight_decay=0.0, grad_clip=1.0, lr_lambda=None, head_lr=None, save_path="best_trainable.pt"):
    params = [p for p in model.parameters() if p.requires_grad]
    head_ids = {id(p) for p in model.head.parameters()}
    groups = [{"params": [p for p in params if id(p) not in head_ids], "lr": lr},
              {"params": [p for p in params if id(p) in head_ids], "lr": head_lr if head_lr is not None else lr}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_per, log = time.time(), -1.0, {}, []
    skipped = 0   # batches whose loss was NaN/inf (skipped, not trained on)
    for epoch in range(epochs):
        model.train(); tot, nb = 0.0, 0
        pbar = tqdm(make_batches(train_ex, batch_size, shuffle=True), desc=f"epoch {epoch + 1}/{epochs}")
        for batch in pbar:
            b = to_tensors(batch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["ids"])
            loss = loss_fn(logits.float(), b)
            if not torch.isfinite(loss):
                if skipped == 0:
                    tqdm.write(f"first non-finite loss at epoch {epoch + 1}, batch {nb + skipped}; "
                               f"max |logit| {logits.float().abs().max().item():.3g}")
                skipped += 1
                opt.zero_grad(set_to_none=True)
                continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", skipped=skipped)
        f1, per = strict_macro_f1(predict(model, val_ex, val_cache, batch_size), val_gold)
        log.append({"epoch": epoch, "train_loss": tot / max(1, nb), "val_macro_f1": f1, "skipped_nonfinite": skipped})
        print(log[-1])
        if f1 > best_f1:
            best_f1, best_per = f1, per
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
    print(f"best val strict macro-F1 {best_f1:.4f} | train time {(time.time()-t0)/60:.1f} min | "
          f"peak GPU mem {torch.cuda.max_memory_allocated()/2**30:.2f} GB")
    for l, (P, R, F1) in best_per.items():
        print(f"  {l:12s} P {P:.3f} R {R:.3f} F1 {F1:.3f}")
    return log, best_f1, best_per


@torch.no_grad()
def time_inference(model, examples, cache, batch_size):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict(model, examples, cache, batch_size)
    torch.cuda.synchronize(); return time.time() - t


def write_submission(model, window, stride, batch_size, best_path=None, out_dir="/kaggle/working"):
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex, test_cache = build_examples(test_text, None, window, stride)
    pred = predict(model, test_ex, test_cache, batch_size)
    out = {d: [[s, e, l] for s, e, l in pred.get(d, [])] for d in test_text}
    assert set(out) == set(test_text), "every fold3 ID, no extras"
    with open(f"{out_dir}/prediction.json", "w") as f:
        json.dump(out, f)
    with zipfile.ZipFile(f"{out_dir}/submission.zip", "w") as z:
        z.write(f"{out_dir}/prediction.json", arcname="prediction.json")
    print("entities predicted:", sum(len(v) for v in out.values()), f"-> {out_dir}/submission.zip")


## Cell E1: full fine-tuning (fill [H], [S], [N])

In [ ]:
# =====================================================================================
# CELL E1 — Full fine-tuning of the c-1 decoder for Legal NER (template)
# Run after Cells 0, A-D and E0. If anything else ran on `lm` in this session,
# re-run Cell A first so training starts from the Phase 1 weights.
#
# YOUR WORK (course rule: no LLM-written architecture or hyperparameters):
#   [H] hyperparameters below
#   [S] which parts of the decoder train (all of it = full FT; some layers = partial FT)
#   [N] the NER head
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters — yours, taken from your c-3 full fine-tune run.
#   WINDOW 200 / STRIDE 10 keep the same step as c-3 (a new window every 10 tokens);
#   here each window labels its last 10 tokens and the 190 before them are left context.
# -------------------------------------------------------------------------------------
WINDOW       = 200    # tokens per window, <= 2048
STRIDE       = 10     # new labelled tokens per window, < WINDOW; WINDOW-STRIDE = left context
LR           = 1e-4   # backbone LR
HEAD_LR      = None   # optional: separate LR for the head (None = same as LR)
EPOCHS       = 10
BATCH_SIZE   = 64     # windows per step; if a T4 runs out of memory, set GRAD_CHECKPOINT = True
WEIGHT_DECAY = 0.0
LR_LAMBDA    = None   # optional: step -> LR multiplier (warm-up / decay)
CLASS_WEIGHTS = None  # optional: torch.tensor([...25...], device=device)
GRAD_CHECKPOINT = False   # memory switch from E0 (not a hyperparameter: same maths, slower)
SEED         = 42
for _n in ["WINDOW", "STRIDE", "LR", "EPOCHS", "BATCH_SIZE"]:
    assert globals()[_n] is not None, f"set {_n}"
assert 0 < STRIDE < WINDOW <= cfg.max_seq_len
random.seed(SEED); torch.manual_seed(SEED)


# -------------------------------------------------------------------------------------
# [S] Which decoder parts train — yours (all of it, as in your c-3 full fine-tune)
#   Parts of c-1: lm.tok_emb (tied with lm.head, so training it also changes the LM head),
#   lm.blocks[0..7] (each: attn.q_proj/k_proj/v_proj/out_proj, ffn.net[0]/[2], ln1, ln2),
#   lm.ln_f. pos_emb is a fixed sinusoidal buffer (nothing to train).
#   Everything starts frozen here.
# -------------------------------------------------------------------------------------
for p in lm.parameters():
    p.requires_grad = False

# Your c-3 choice: train everything (full fine-tuning)
# Train token embeddings (tied with lm.head)
for p in lm.tok_emb.parameters():
    p.requires_grad = True

# Train all 8 decoder blocks
for p in lm.blocks.parameters():
    p.requires_grad = True

# Train the final layer normalization
for p in lm.ln_f.parameters():
    p.requires_grad = True


# -------------------------------------------------------------------------------------
# [N] Head — yours, from your c-3 full fine-tune: one linear layer FEAT_DIM (1024) -> N_TAGS (25)
# -------------------------------------------------------------------------------------
head = nn.Linear(FEAT_DIM, N_TAGS)
assert head is not None, "define the head"
model = NERModel(lm, head).to(device)
count_params(model)
assert any(p.requires_grad for p in lm.parameters()), "no decoder parameter is trainable: set [S]"


# -------------------------------------------------------------------------------------
# Data, training, scoring (plumbing from E0)
# -------------------------------------------------------------------------------------
train_ex, _ = build_examples(f1_text, f1_gold, WINDOW, STRIDE)
val_ex, val_cache = build_examples(f2_text, f2_gold, WINDOW, STRIDE)
print(f"train windows {len(train_ex)} | val windows {len(val_ex)}")

log, best_f1, per = train(
    model, train_ex, val_ex, val_cache, f2_gold,
    loss_fn=lambda logits, b: ce_loss(logits, b["labels"], CLASS_WEIGHTS),
    lr=LR, head_lr=HEAD_LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path="/kaggle/working/c1_fullft_best_trainable.pt")
print(f"fold2 inference time {time_inference(model, val_ex, val_cache, BATCH_SIZE):.1f} s")

write_submission(model, WINDOW, STRIDE, BATCH_SIZE, best_path="/kaggle/working/c1_fullft_best_trainable.pt")


## Cell E2: LoRA (fill [H]; [L] and [N] carry your c-6 code). Can run right after E1: it reloads fresh Phase 1 weights itself.

In [11]:
# # =====================================================================================
# # CELL E2 — LoRA on the c-1 decoder for Legal NER
# # Run after Cells 0, A-D and E0. It may run straight after E1: step 1 below throws away
# # E1's fine-tuned model and reloads the Phase 1 c-1 weights, so LoRA starts from the
# # same backbone as E1 did. (E1's best weights stay saved in c1_fullft_best_trainable.pt.)
# #
# # YOUR WORK (course rule: no LLM-written architecture or hyperparameters):
# #   [H] hyperparameters below
# #   [L] LoRALinear: the low-rank update itself — ref [3] Hu et al. 2021 (your c-6 code)
# #   [N] the NER head (your c-6 code)
# # =====================================================================================

# # -------------------------------------------------------------------------------------
# # [H] Hyperparameters — all yours. None = not set; the assert stops you.
# #   All values are from your c-6 LoRA run. WINDOW 200 / STRIDE 10 keep the same step as
# #   c-6 (a new window every 10 tokens); here each window labels its last 10 tokens and the
# #   190 before them are left context.
# # -------------------------------------------------------------------------------------
# WINDOW       = 200    # tokens per window, <= 2048
# STRIDE       = 10     # new labelled tokens per window, < WINDOW; WINDOW-STRIDE = left context
# LORA_RANK    = 200    # your c-6 value (r)
# LORA_ALPHA   = 50     # your c-6 value (update is scaled by alpha / r)
# LORA_TARGETS = ("ffn_out",)   # your c-6 setting; subset of
#                               # ("q_proj", "k_proj", "v_proj", "out_proj", "ffn_in", "ffn_out")
# LORA_LAYERS  = tuple(range(cfg.n_layers))   # which of the 8 blocks get LoRA (all by default)
# LR           = 1e-4   # your c-6 value
# HEAD_LR      = None   # optional: separate LR for the head (None = same as LR)
# EPOCHS       = 10     # your c-6 value
# BATCH_SIZE   = 64     # your c-6 value; windows per step
# WEIGHT_DECAY = 0.0
# LR_LAMBDA    = None   # optional: step -> LR multiplier (warm-up / decay)
# CLASS_WEIGHTS = None  # optional: torch.tensor([...25...], device=device)
# GRAD_CHECKPOINT = False   # memory switch from E0 (same maths, slower)
# SEED         = 42
# for _n in ["WINDOW", "STRIDE", "LORA_RANK", "LORA_ALPHA", "LORA_TARGETS", "LR", "EPOCHS", "BATCH_SIZE"]:
#     assert globals()[_n] is not None, f"set {_n}"
# assert 0 < STRIDE < WINDOW <= cfg.max_seq_len
# random.seed(SEED); torch.manual_seed(SEED)


# # -------------------------------------------------------------------------------------
# # 1. Fresh Phase 1 c-1 backbone (drops whatever E1 trained)
# # -------------------------------------------------------------------------------------
# for _name in ("model", "head", "train_ex", "val_ex"):
#     globals().pop(_name, None)
# del lm; import gc; gc.collect(); torch.cuda.empty_cache()
# _ck = torch.load(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt", map_location="cpu", weights_only=False)
# lm = DecoderLM(LMConfig(**_ck["config"])); print(lm.load_state_dict(_ck["model"])); lm.to(device).eval()
# del _ck


# # -------------------------------------------------------------------------------------
# # [L] LoRA layer — yours, copied from your c-6 cell (unchanged)
# #   Note from c-6: A ~ N(0, 1) was stable with ffn_out only, but went NaN under fp16 when
# #   all four targets were wrapped. If you widen LORA_TARGETS, revisit the init scale of A.
# # -------------------------------------------------------------------------------------
# class LoRALinear(nn.Module):
#     def __init__(self, base: nn.Linear, rank: int, alpha: float):
#         super().__init__()
#         self.base = base
#         for p in self.base.parameters():
#             p.requires_grad = False
#         self.in_features, self.out_features = base.in_features, base.out_features
#         self.rank, self.alpha = rank, alpha
#         self.A = nn.Linear(self.in_features, rank, bias=False)
#         self.B = nn.Linear(rank, self.out_features, bias=False)

#         nn.init.normal_(self.A.weight, mean=0.0, std=1.0)
#         nn.init.zeros_(self.B.weight)

#         self.scaling = alpha / rank
#     def forward(self, x):
#         return self.base(x) + self.scaling * self.B(self.A(x))


# # -------------------------------------------------------------------------------------
# # 2. Freeze c-1, wrap the chosen linears, check identity at init (plumbing)
# # -------------------------------------------------------------------------------------
# def decoder_linears(backbone, which, layers):
#     """Yield (parent, attr, nn.Linear) in the chosen c-1 blocks.
#     which ⊆ {"q_proj", "k_proj", "v_proj", "out_proj", "ffn_in", "ffn_out"}."""
#     for li in layers:
#         blk = backbone.blocks[li]
#         for w in which:
#             if w in ("q_proj", "k_proj", "v_proj", "out_proj"):
#                 parent, attr = blk.attn, w
#             elif w == "ffn_in":
#                 parent, attr = blk.ffn.net, "0"     # 1024 -> 4096
#             elif w == "ffn_out":
#                 parent, attr = blk.ffn.net, "2"     # 4096 -> 1024
#             else:
#                 raise ValueError(w)
#             yield parent, attr, getattr(parent, attr)

# probe_text = next(t for t in f1_text.values() if len(encode_with_offsets(t)[0]) >= 64)
# probe = torch.tensor([encode_with_offsets(probe_text)[0][:64]], device=device)
# with torch.no_grad():
#     ref = features(probe).float()

# for p in lm.parameters():
#     p.requires_grad = False
# todo = list(decoder_linears(lm, LORA_TARGETS, LORA_LAYERS))
# for parent, attr, lin in todo:
#     assert isinstance(lin, nn.Linear), f"{attr} is already wrapped"
#     setattr(parent, attr, LoRALinear(lin, LORA_RANK, LORA_ALPHA).to(device))

# with torch.no_grad():
#     diff = (features(probe).float() - ref).abs().max().item()
# print(f"wrapped {len(todo)} linears | LoRA identity-at-init max |diff| = {diff:.2e} (should be ~0)")


# # -------------------------------------------------------------------------------------
# # [N] Head — yours, copied from your c-6 cell (FEAT_DIM is 1024 for c-1)
# # -------------------------------------------------------------------------------------
# hdim = 200
# head = nn.Sequential(
#     nn.Linear(FEAT_DIM, hdim),
#     nn.ReLU(),
#     nn.Linear(hdim, N_TAGS)
# )
# assert head is not None, "define the head"
# model = NERModel(lm, head).to(device)
# count_params(model)


# # -------------------------------------------------------------------------------------
# # Data, training, scoring (plumbing from E0)
# # -------------------------------------------------------------------------------------
# train_ex, _ = build_examples(f1_text, f1_gold, WINDOW, STRIDE)
# val_ex, val_cache = build_examples(f2_text, f2_gold, WINDOW, STRIDE)
# print(f"train windows {len(train_ex)} | val windows {len(val_ex)}")

# log, best_f1, per = train(
#     model, train_ex, val_ex, val_cache, f2_gold,
#     loss_fn=lambda logits, b: ce_loss(logits, b["labels"], CLASS_WEIGHTS),
#     lr=LR, head_lr=HEAD_LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
#     lr_lambda=LR_LAMBDA, save_path="/kaggle/working/c1_lora_best_trainable.pt")
# print(f"fold2 inference time {time_inference(model, val_ex, val_cache, BATCH_SIZE):.1f} s")

# # Overwrites E1's /kaggle/working/submission.zip: download E1's first if you want to keep both
# write_submission(model, WINDOW, STRIDE, BATCH_SIZE, best_path="/kaggle/working/c1_lora_best_trainable.pt")


<All keys matched successfully>
wrapped 8 linears | LoRA identity-at-init max |diff| = 0.00e+00 (should be ~0)
trainable 8.402M / total 119.38M (7.04%)


tokenising + windowing:   0%|          | 0/35 [00:00<?, ?it/s]

tokenising + windowing:   0%|          | 0/35 [00:00<?, ?it/s]

train windows 24481 | val windows 24519


epoch 1/10:   0%|          | 0/383 [00:00<?, ?it/s]

predict:   0%|          | 0/384 [00:00<?, ?it/s]

... (29 outputs left out)

fold2 inference time 93.0 s


tokenising + windowing:   0%|          | 0/35 [00:00<?, ?it/s]

predict:   0%|          | 0/412 [00:00<?, ?it/s]

entities predicted: 7962 -> /kaggle/working/submission.zip
